In [ ]:
import pandas as pd
import numpy as np

URL = "https://physionet-open.s3.amazonaws.com/gaitpdb/1.0.0/"

dem = pd.read_csv(URL + "demographics.txt", sep="\t", usecols=range(20)).dropna(subset=["ID"])
ga = dem[(dem.Study == "Ga") & dem.Speed_01.notna()]

print("Grup başına kişi sayısı:", ga.groupby("Group").size().to_dict())
print("Parkinson ID'leri Pt ile başlıyor:", ga[ga.Group == 1].ID.str.contains("Pt").all())

def oku(ID):
    return pd.read_csv(URL + ID + "_01.txt", sep="\t", header=None).values

for ID in ["GaPt04", "GaCo01"]:
    d = oku(ID)
    print(ID, "shape:", d.shape)
    print(ID, "medyan zaman aralığı (s):", np.median(np.diff(d[:, 0])))
    print(ID, "sağlama (N):", np.abs(d[:, 1:9].sum(axis=1) - d[:, 17]).max())

In [ ]:
import numpy as np

esik_N = 20.0

def sol_tek_destek_sureleri(ID):
    d = oku(ID)
    zaman = d[:, 0]
    sag_kuvvet = d[:, 18]
    sag_yerde = sag_kuvvet >= esik_N

    sag_bas_idx = np.flatnonzero((~sag_yerde[:-1]) & sag_yerde[1:]) + 1
    sag_kalk_idx = np.flatnonzero(sag_yerde[:-1] & (~sag_yerde[1:])) + 1

    sureler = []
    for kalk in sag_kalk_idx:
        sonraki_baslar = sag_bas_idx[sag_bas_idx > kalk]
        if sonraki_baslar.size > 0:
            bas = sonraki_baslar[0]
            sureler.append(zaman[bas] - zaman[kalk])

    print(f"{ID}: sağ basış={len(sag_bas_idx)}, sağ parmak kalkış={len(sag_kalk_idx)}, tamamlanan sol tek destek={len(sureler)}")
    print("İlk 5 sol tek destek süresi (s):", np.round(sureler[:5], 3))
    return np.array(sureler)

ornek_sureler = {
    ID: sol_tek_destek_sureleri(ID)
    for ID in ["GaPt04", "GaCo01"]
}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = oku("GaCo01")
zaman = d[:, 0]
sag_kuvvet = d[:, 18]
sag_yerde = sag_kuvvet >= 20.0

sag_bas_idx = np.flatnonzero((~sag_yerde[:-1]) & sag_yerde[1:]) + 1
sag_kalk_idx = np.flatnonzero(sag_yerde[:-1] & (~sag_yerde[1:])) + 1

kalk_idx = sag_kalk_idx[0]
bas_idx = sag_bas_idx[sag_bas_idx > kalk_idx][0]

gorunen_aralik = (zaman >= zaman[kalk_idx] - 0.25) & (zaman <= zaman[bas_idx] + 0.25)

plt.figure(figsize=(10, 4))
plt.plot(zaman[gorunen_aralik], sag_kuvvet[gorunen_aralik], label="Sağ ayak kuvveti")
plt.axhline(20.0, color="gray", linestyle="--", label="20 N eşiği")
plt.axvline(zaman[kalk_idx], color="red", linestyle=":", label="Sağ parmak kalkışı")
plt.axvline(zaman[bas_idx], color="green", linestyle=":", label="Sonraki sağ basış")
plt.xlabel("Zaman (s)")
plt.ylabel("Kuvvet (N)")
plt.title("GaCo01: ilk sol tek destek aralığının kontrolü")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
sonuclar = []
esik_N = 20.0

for ID, grup, hiz in ga[["ID", "Group", "Speed_01"]].itertuples(index=False, name=None):
    d = oku(ID)
    zaman = d[:, 0]
    sag_yerde = d[:, 18] >= esik_N

    bas_idx = np.flatnonzero((~sag_yerde[:-1]) & sag_yerde[1:]) + 1
    kalk_idx = np.flatnonzero(sag_yerde[:-1] & (~sag_yerde[1:])) + 1

    sureler = []
    j = 0
    for kalk in kalk_idx:
        while j < len(bas_idx) and bas_idx[j] <= kalk:
            j += 1
        if j < len(bas_idx):
            sureler.append(zaman[bas_idx[j]] - zaman[kalk])
            j += 1

    sureler = np.asarray(sureler, dtype=float)
    ortalama = np.mean(sureler)
    sd = np.std(sureler, ddof=1)
    cv = 100 * sd / ortalama

    sonuclar.append([ID, grup, hiz, len(sureler), ortalama, sd, cv])

kisi_cv = pd.DataFrame(
    sonuclar,
    columns=["ID", "Group", "Speed_01", "aralik_sayisi", "ortalama_s", "sd_s", "CV_yuzde"]
)

print("Kişi sayısı:", len(kisi_cv))
display(kisi_cv.round(3))

In [ ]:
from scipy.stats import t as t_dist

pd_cv = kisi_cv.loc[kisi_cv["Group"] == 1.0, "CV_yuzde"]
kontrol_cv = kisi_cv.loc[kisi_cv["Group"] == 2.0, "CV_yuzde"]
n_pd = len(pd_cv)
n_kontrol = len(kontrol_cv)
fark = pd_cv.mean() - kontrol_cv.mean()
var_pd = pd_cv.var(ddof=1)
var_kontrol = kontrol_cv.var(ddof=1)
a = var_pd / n_pd
b = var_kontrol / n_kontrol
se = np.sqrt(a + b)
df = (a + b) ** 2 / (a**2 / (n_pd - 1) + b**2 / (n_kontrol - 1))
t_kritik = t_dist.ppf(0.975, df)
alt = fark - t_kritik * se
ust = fark + t_kritik * se

print("Parkinson ortalama CV (%):", round(pd_cv.mean(), 3))
print("Kontrol ortalama CV (%):", round(kontrol_cv.mean(), 3))
print("Ham fark (yüzde puan):", round(fark, 3))
print("Welch %95 güven aralığı (yüzde puan):", (round(alt, 3), round(ust, 3)))

In [ ]:
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

kisi_cv = kisi_cv.copy()
kisi_cv["PD"] = (kisi_cv["Group"] == 1.0).astype(int)
model_hiz = smf.ols("CV_yuzde ~ PD + Speed_01", data=kisi_cv).fit(cov_type="HC3", use_t=True)
hizli_fark = model_hiz.params["PD"]
hizli_alt, hizli_ust = model_hiz.conf_int().loc["PD"].to_numpy()

print("Hız ayarlı fark (yüzde puan):", round(hizli_fark, 3))
print("Hız ayarlı %95 güven aralığı (yüzde puan):", (round(hizli_alt, 3), round(hizli_ust, 3)))

def fark_grafigi(tahmin, alt, ust, baslik, dosya):
    fig, ax = plt.subplots(figsize=(6.5, 2.8))
    ax.errorbar(tahmin, 0, xerr=[[tahmin - alt], [ust - tahmin]], fmt="o", capsize=6, markersize=8)
    ax.axvline(0, color="gray", linestyle="--")
    ax.set_yticks([0])
    ax.set_yticklabels(["Parkinson − kontrol"])
    ax.set_xlabel("CV farkı (yüzde puan)")
    ax.set_title(baslik)
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    fig.savefig(dosya, dpi=200, bbox_inches="tight")
    plt.show()

fark_grafigi(fark, alt, ust, "Ham tek destek süresi CV farkı", "sekil1.png")
fark_grafigi(hizli_fark, hizli_alt, hizli_ust, "Hız ayarlı tek destek süresi CV farkı", "sekil2.png")
print("Kaydedilen dosyalar: sekil1.png, sekil2.png")

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

# Colab çalışma klasöründe iki PNG'nin bulunduğunu kontrol et.
klasor = Path.cwd()
ciktilar = [klasor / "sekil1.png", klasor / "sekil2.png"]
eksikler = [yol.name for yol in ciktilar if not yol.is_file()]
if eksikler:
    raise FileNotFoundError(f"Eksik çıktı dosyaları: {eksikler}")

# Kişi başına türetilmiş ölçü tablosunu kaydet; ham kuvvet verisi ekleme.
csv_yolu = klasor / "tablo1.csv"
kisi_cv.drop(columns=["PD"], errors="ignore").to_csv(csv_yolu, index=False, float_format="%.6f")

# Kaynak ve lisans bilgisini ayrı bir metin dosyasına yaz.
atif_yolu = klasor / "KAYNAK_ATIF.txt"
atif_yolu.write_text(
    "Kaynak: PhysioNet Gait in Parkinson's Disease (GaitPDB), sürüm 1.0.0.\n"
    "https://physionet.org/content/gaitpdb/1.0.0/\n"
    "Bu tablo Ga grubundaki Speed_01 kaydından türetilmiştir. Katılımcı kodları veri setindeki kodlardır.\n"
    "Kaynak veri lisansı: Open Data Commons Attribution License (ODC-By) 1.0.\n"
    "Paket yalnız türetilmiş tabloyu ve şekilleri içerir; ham kuvvet dosyaları içermez.\n",
    encoding="utf-8",
)

# İki şekli, CSV'yi ve atıf notunu tek ZIP'e koy.
zip_yolu = klasor / "Parkinson_CV_ciktilari.zip"
with zipfile.ZipFile(zip_yolu, "w", compression=zipfile.ZIP_DEFLATED) as paket:
    for yol in ciktilar + [csv_yolu, atif_yolu]:
        paket.write(yol, arcname=yol.name)

# İndirme öncesi dosya adlarını, tablo satır sayısını ve boyutları göster.
print("Paket içeriği:", [yol.name for yol in ciktilar + [csv_yolu, atif_yolu]])
print("Tablo kişi satırı:", len(kisi_cv))
print("Tablo sütunları:", list(kisi_cv.drop(columns=["PD"], errors="ignore").columns))
print("PNG boyutları (bayt):", {yol.name: yol.stat().st_size for yol in ciktilar})
print("ZIP boyutu (bayt):", zip_yolu.stat().st_size)

# ZIP dosyasını bu bilgisayara indir.
files.download(str(zip_yolu))